In [18]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import json
from pathlib import Path
from typing import Any

hf_token =json.load(open(Path("./config.json")))["hg_access_token"]
cache_dir =json.load(open(Path("./config.json")))["cache_dir"]

In [19]:
def parse_json_value(value: Any) -> dict:
    """
    Convert either a JSON string or Python dictionary into a dictionary.
    """
    if isinstance(value, dict):
        return value

    if isinstance(value, str):
        parsed = json.loads(value)

        if not isinstance(parsed, dict):
            raise ValueError("Tool-call output must be a JSON object.")

        return parsed

    raise TypeError(
        f"Expected output to be dict or JSON string, got {type(value).__name__}"
    )


In [4]:
class QueryGenerator:
    def __init__(self, model, tokenizer, system_prompt, max_new_tokens=512, temperature=2.0):
        self.model = model
        self.tokenizer = tokenizer
        self.system_prompt = system_prompt
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature

    def build_messages(self, user_prompt):
        return [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": user_prompt},
        ]

    def build_text(self, user_prompt):
        messages = self.build_messages(user_prompt)

        return self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            enable_thinking=False,
            add_generation_prompt=True,
        )

    def generate(self, user_prompt):
        text = self.build_text(user_prompt)

        model_inputs = self.tokenizer(
            [text],
            return_tensors="pt",
        ).to(self.model.device)

        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=self.max_new_tokens,
            temperature=self.temperature,
            top_p= 0.8,
            top_k=20,
            min_p=0 
            
        )

        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

        model_output = self.tokenizer.decode(
            output_ids,
            skip_special_tokens=True,
        ).strip("\n")

        return model_output



In [5]:
SYSTEM_PROMPT = """
You are a financial function-calling model.
The user gives you names of companiesn, some feature and periods of time. You must convert the user's request into exactly one valid JSON object.

Supported function:
- get_fundamentals

Rules:

- Preserve the association between companies and their requested metrics.
- Do not answer the financial question yourself, analysis, speculations, or hypothetical questpions.
- Do not calculate or invent financial values.
- Do not generate SQL.
- Output only valid JSON.
""".strip()


In [6]:
model_path = "./models/qwen3_0_6b_tool_calling_lora/"
# model_name = "meta-llama/Llama-3.2-3B-Instruct"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    token=hf_token,
    cache_dir=cache_dir,
    torch_dtype="auto",
    device_map="auto",
    
)


Loading weights: 100%|██████████| 112/112 [00:00<00:00, 14473.81it/s]


In [7]:
query_generator= QueryGenerator(model,tokenizer,SYSTEM_PROMPT, max_new_tokens=328,temperature=0.7)


In [52]:
user_prompt ="From 2020 to 2024 bring : revenue of Amazon, net income of tesla"


In [53]:

model_output = query_generator.generate(user_prompt)
print(model_output)

{"action":"call","function":"get_fundamentals","arguments":{"queries":[{"symbols":["Tesla"],"metrics":["Net Income"],"start_year":2020,"end_year":2024},{"symbols":["Amazon"],"metrics":["Revenue"],"start_year":2020,"end_year":2024}]}}


In [54]:
parsed_output= parse_json_value(model_output)

In [55]:
parsed_output['arguments']['queries']

[{'symbols': ['Tesla'],
  'metrics': ['Net Income'],
  'start_year': 2020,
  'end_year': 2024},
 {'symbols': ['Amazon'],
  'metrics': ['Revenue'],
  'start_year': 2020,
  'end_year': 2024}]

In [10]:
user_prompt_1 ="what is the weather"

model_output_1 = query_generator.generate(user_prompt_1)
print(model_output_1)

{"action":"call","function":"get_fundamentals","arguments":{"queries":[{"symbols":[""],"metrics":["Weather"],"start_year":1900,"end_year":2023}]}}
